In [0]:
from pyspark.sql import functions as F

df = spark.table("workspace.aml_bronze.raw_transactions")

unique_accounts = (
    df.select(F.col("from_account").alias("account_id"))
    .union(df.select(F.col("to_account").alias("account_id")))
    .distinct()
)

countries_data = [
    ("US", "United States", "Low"), ("GB", "United Kingdom", "Low"),
    ("DE", "Germany", "Low"), ("FR", "France", "Low"),
    ("CH", "Switzerland", "Medium"), ("AE", "UAE", "Medium"),
    ("CY", "Cyprus", "Medium"), ("PA", "Panama", "High"),
    ("KY", "Cayman Islands", "High"), ("IR", "Iran", "High"),
    ("KP", "North Korea", "High"),
]
countries_df = spark.createDataFrame(countries_data, schema=["country_code", "country_name", "risk_rating"])

country_codes = ["US", "GB", "DE", "FR", "CH", "AE", "CY", "PA", "KY", "IR", "KP"]
occupations = ["Engineer", "Retail", "Finance", "Consulting", "Import/Export", "Real Estate"]

customers_df = (
    unique_accounts
    .withColumn("_rand_country", (F.rand(seed=42) * len(country_codes)).cast("int"))
    .withColumn("_rand_occupation", (F.rand(seed=99) * len(occupations)).cast("int"))
    .withColumn("_rand_pep", F.rand(seed=7))
    .withColumn("country_code", F.element_at(F.array([F.lit(c) for c in country_codes]), F.col("_rand_country") + 1))
    .withColumn("occupation", F.element_at(F.array([F.lit(o) for o in occupations]), F.col("_rand_occupation") + 1))
    .withColumn("is_pep", F.when(F.col("_rand_pep") < 0.02, 1).otherwise(0))
    .drop("_rand_country", "_rand_occupation", "_rand_pep")
)

In [0]:
from faker import Faker
import pandas as pd

fake = Faker()
Faker.seed(42)

account_ids = [row.account_id for row in unique_accounts.select("account_id").collect()]

names_data = [(acc_id, fake.first_name(), fake.last_name()) for acc_id in account_ids]
names_pdf = pd.DataFrame(names_data, columns=["account_id", "first_name", "last_name"])
names_df = spark.createDataFrame(names_pdf)

names_df.show(5)

In [0]:
customers_df = customers_df.join(names_df, on="account_id", how="left")
customers_df.select("account_id", "first_name", "last_name", "country_code", "occupation", "is_pep").show(10)

In [0]:
(
    customers_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.aml_bronze.customers")
)

(
    countries_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.aml_bronze.countries")
)

In [0]:
transactions = spark.table("workspace.aml_silver.stg_transactions")
customers = spark.table("workspace.aml_bronze.customers")
countries = spark.table("workspace.aml_bronze.countries")

enriched = (
    transactions
    .join(customers, transactions.from_account == customers.account_id, "left")
    .join(countries, customers.country_code == countries.country_code, "left")
    .select(
        transactions["*"],
        customers["first_name"],
        customers["last_name"],
        customers["occupation"],
        customers["is_pep"],
        countries["country_name"],
        countries["risk_rating"].alias("sender_country_risk")
    )
)

enriched.select("from_account", "first_name", "last_name", "country_name", "sender_country_risk", "is_pep", "amount_paid").show(10)

In [0]:
enriched.groupBy("sender_country_risk").agg(
    F.count("*").alias("total_txns"),
    F.sum("is_laundering").alias("laundering_txns"),
    (F.sum("is_laundering") / F.count("*") * 100).alias("laundering_rate_pct")
).orderBy(F.desc("laundering_rate_pct")).show()